# KernelForge — GPU session

**Settings → Accelerator → GPU T4** and **Internet → On**, then *Run All*.

Produces the GPU-verified dataset (`data/verified/dataset.jsonl`), the per-category pass report,
and a zero-shot baseline eval on every held-out prompt. Set `FINETUNE = True` to also train the
QLoRA adapter and run the fine-tuned eval (adds about 1 hour).

**All outputs land in `/kaggle/working/kernelforge_outputs.zip`** (Output panel → download after the
session ends). The zip is refreshed after every stage and every 10 minutes during the dataset stage,
so a disconnect never loses finished work.

**Resuming after a disconnect:**
1. Upload the old `kernelforge_outputs.zip` as a Kaggle Dataset.
2. Add it to this notebook (Add Data → Your Datasets).
3. Set `RESUME_FROM` below to its `/kaggle/input/<name>/kernelforge_outputs.zip` path.
4. Run All — stages already finished are skipped automatically.

See `docs/handoff/KAGGLE_RUNBOOK.md` for a full reference.

In [ ]:
# --- Session settings ---
FINETUNE     = False   # True: + QLoRA fine-tune and fine-tuned eval (~1 h extra)
EVAL_LIMIT   = None    # None = all 100 held-out prompts; integer caps it round-robin
BRANCH       = "main"
BUDGET_HOURS = 11      # stop cleanly before Kaggle's 12 h wall-clock limit
RESUME_FROM  = ""      # "/kaggle/input/kf-prev/kernelforge_outputs.zip" to resume

# --- Approximate time budget on a single T4 (untested; revise as you measure) ---
# Stage          Est. time   Notes
# env            < 1 min     nvidia-smi, python/triton version check
# harness        1–2 min     smoke-checks the verifier against one kernel
# dataset        4–5 h       300 kernels × (compile + verify); 15 may OOM (counted, not fatal)
# baseline       1–2 h       100 prompts, greedy, skip-verification=False, 2048 max tokens
# finetune       ~1 h        QLoRA, 1.5B, 3 epochs, batch=1, grad_accum=8  (FINETUNE=True)
# eval           ~1 h        100 prompts on the fine-tuned adapter          (FINETUNE=True)
# Total (no FT)  ~6–7 h      well within 12 h
# Total (+ FT)   ~8–9 h      still within 12 h


In [ ]:
# Cell 2a: environment check + clone + install
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!rm -rf KernelForge && git clone -q --depth 1 -b {BRANCH} https://github.com/Arsalan-05/KernelForge.git
%cd KernelForge
!pip install -q -r requirements-demo.txt
!python -c "import torch, triton; print('torch', torch.__version__, '| triton', triton.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# Cell 2b: pre-flight check — must end with 0 failures before spending GPU time
!python scripts/preflight_check.py
# Expected output: 'All required checks passed.'
# If 'required check(s) failed' appears, fix the error above before proceeding.

In [ ]:
# Cell 2c: split-leakage check — confirm no train/test contamination
!python scripts/check_split_leakage.py
# Expected output: 'PASSED — no leakage detected.'

In [ ]:
# Cell 3: run the full GPU session
import shlex
flags = f"--budget-hours {BUDGET_HOURS}"
flags += f" --eval-limit {EVAL_LIMIT}" if EVAL_LIMIT else ""
flags += " --finetune" if FINETUNE else ""
flags += f" --resume-from {shlex.quote(RESUME_FROM)}" if RESUME_FROM else ""
# gpu_session.py zips outputs after every stage and sets a non-zero exit if any stage
# failed; a failure does NOT stop later stages (they run and may still produce output).
!python scripts/gpu_session.py {flags}
print("\nAll stages attempted. See session summary in cell 4.")

In [ ]:
import json, os
path = "/kaggle/working/kernelforge_outputs/session.json"
if not os.path.exists(path):
    print("No session.json: gpu_session.py did not get far enough to write one (see its output above).")
else:
    s = json.load(open(path))
    print(json.dumps(s.get("env", {}), indent=2))
    for stage, r in s.get("stages", {}).items():
        print(f"{stage:<10} {r['status']:<8} {r.get('seconds', 0):>8}s  {r.get('error') or ''}")
        if "prompts_evaluated" in r:
            print(f"{'':<10} {r['prompts_evaluated']} of {r['held_out_total']} held-out prompts "
                  f"({'full set' if r['full_held_out_set'] else 'NOT the full set'})")
    rep = s.get("dataset_report", {})
    if rep.get("partial"):
        print(f"\nDataset run is PARTIAL: {len(rep['run_stats']['not_run'])} entries not run (resume to finish).")
    for cat, r in rep.get("per_category", {}).items():
        print(f"{cat:<18} {r}")

In [ ]:
# Cell 5: fallback — download whatever is in /kaggle/working if the zip is missing
import os, shutil
out = "/kaggle/working/kernelforge_outputs"
zip_path = out + ".zip"
if os.path.exists(zip_path):
    print("Download:", zip_path)
else:
    print("No zip found — building a fallback zip from whatever is in", out)
    os.makedirs(out, exist_ok=True)
    for rel in ["data/verified", "data/raw", "evaluation/results", "training/checkpoints"]:
        if os.path.isdir(rel):
            shutil.copytree(rel, os.path.join(out, rel), dirs_exist_ok=True)
            print(" copied", rel)
    result = shutil.make_archive(out, "zip", root_dir=out)
    print("Fallback zip:", result)